
# 練習問題: CPU と GPU はどちらが速い? (損益分岐点を探す)

## 目標

GPU は「十分に大きな並列計算」では CPU より速いが, 仕事が小さいとデータ転送や起動の overhead に負けて **CPU の方が速い** こともある. 同じプログラムを CPU と GPU で動かし, GPU が勝ち始める問題サイズ (損益分岐点) を体験する.

## 課題

`cpu_vs_gpu.cpp` (または `cpu_vs_gpu.f90`) は, `lin_rec` (`x = a*x + b` を `n` 回) を `m` 個の独立な要素について計算し, GFLOPS を表示する.
このループには **すでにGPU用のオフロード指示が付いている** (穴埋めは不要).

同じ実行ファイルを, 環境変数 `OMP_TARGET_OFFLOAD` で実行先を切り替えて比較する:

- `OMP_TARGET_OFFLOAD=DISABLED` … ホスト (CPU) で実行
- `OMP_TARGET_OFFLOAD=MANDATORY` … デバイス (GPU) で実行 (GPUが無ければエラー)

問題サイズ `m` を小さい値から大きい値まで変えながら両方を測り, **どのあたりの `m` で GPU が CPU を逆転するか** を調べよ. (CPU実行時は `OMP_NUM_THREADS` でスレッド数も指定できる.)

## コンパイルと実行

```
# C++
nvc++ -fast -mp=gpu cpu_vs_gpu.cpp -o cpu_vs_gpu.exe

# Fortran
nvfortran -fast -mp=gpu cpu_vs_gpu.f90 -o cpu_vs_gpu.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入する.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

for m in 1 8 64 512 4096 32768 262144 ; do
    echo -n "CPU m=$m : "
    OMP_TARGET_OFFLOAD=DISABLED OMP_NUM_THREADS=9 ./cpu_vs_gpu.exe $m | grep GFLOPS
    echo -n "GPU m=$m : "
    OMP_TARGET_OFFLOAD=MANDATORY ./cpu_vs_gpu.exe $m | grep GFLOPS
done
```

## 期待される結果

- `m` が小さいうちは CPU の方が GFLOPS が高い (GPU は並列度を使い切れず, overhead が目立つ).
- `m` を大きくしていくと, あるサイズを境に GPU が CPU を上回り, さらに大きくすると差が開く.

「GPU は何でも速い」わけではなく, **十分な仕事量があって初めて GPU が活きる** ことを, 損益分岐点の `m` の値とともに確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ cpu_vs_gpu.cpp
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* x = a*x + b を n 回繰り返す (2n flops). m 要素を独立に計算する. */
double lin_rec(double a, double b, double c, long n) {
  double x = c;
  for (long j = 0; j < n; j++) x = a * x + b;
  return x;
}

int main(int argc, char ** argv) {
  long m = (1 < argc ? atol(argv[1]) : 1024);
  long n = (2 < argc ? atol(argv[2]) : 1000 * 1000);
  double * x = (double *)calloc(sizeof(double), m);
  assert(x);

  double t0 = omp_get_wtime();
  /* このループは GPU 用にオフロード指示が付いている (完成済み).
     OMP_TARGET_OFFLOAD=DISABLED ならホスト(CPU)で, MANDATORY ならGPUで実行される. */
#pragma omp target teams distribute parallel for map(tofrom: x[0:m])
  for (long i = 0; i < m; i++) {
    x[i] = lin_rec(0.99, i + 1, 1.0, n);
  }
  double t1 = omp_get_wtime();
  double dt = t1 - t0;

  double flops = 2.0 * (double)m * (double)n;
  printf("m = %ld, n = %ld, elapsed = %.3f sec, %.3f GFLOPS\n",
         m, n, dt, flops / dt * 1e-9);
  free(x);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu cpu_vs_gpu.cpp -o cpu_vs_gpu_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./cpu_vs_gpu_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ cpu_vs_gpu.f90
module lin_rec_mod
contains
  ! x = a*x + b を n 回繰り返す (2n flops)
  function lin_rec(a, b, c, n) result(x)
    !$omp declare target
    real(8), intent(in) :: a, b, c
    integer(8), intent(in) :: n
    real(8) :: x
    integer(8) :: j
    x = c
    do j = 1, n
       x = a * x + b
    end do
  end function lin_rec
end module lin_rec_mod

program cpu_vs_gpu
  use omp_lib
  use lin_rec_mod
  character(len=32) :: arg
  integer(8) :: m, n, i
  real(8), allocatable :: x(:)
  real(8) :: t0, t1, dt, flops
  m = 1024
  n = 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  end if
  allocate(x(m))

  t0 = omp_get_wtime()
  ! このループは GPU 用にオフロード指示が付いている (完成済み).
  ! OMP_TARGET_OFFLOAD=DISABLED ならホスト(CPU)で, MANDATORY ならGPUで実行される.
  !$omp target teams distribute parallel do map(tofrom: x)
  do i = 1, m
     x(i) = lin_rec(0.99d0, real(i, 8), 1.0d0, n)
  end do
  !$omp end target teams distribute parallel do
  t1 = omp_get_wtime()
  dt = t1 - t0

  flops = 2.0d0 * real(m, 8) * real(n, 8)
  print "(a,i0,a,i0,a,f0.3,a,f0.3,a)", &
       "m = ", m, ", n = ", n, ", elapsed = ", dt, " sec, ", flops / dt * 1e-9, " GFLOPS"
end program cpu_vs_gpu

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu cpu_vs_gpu.f90 -o cpu_vs_gpu_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./cpu_vs_gpu_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:cpu_vs_gpu.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:cpu_vs_gpu.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:cpu_vs_gpu.cpp}